## TIFF plot

In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")


In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)


In [ ]:
global_topo_tiff_gz <- "global_topo.tiff.gz"

# nchar(global_topo_tiff)
filename.length <- nchar(global_topo_tiff_gz)

# Get the name without the extension ("." + 2 letters)
start <- 1
end <- filename.length-3
global_topo_tiff <- substr(global_topo_tiff_gz,start, end)

# ==============================================================================

if(!file.exists(global_topo_tiff_gz)){
    download.file("https://topex.ucsd.edu/pub/global_topo_tiff/global.tiff.gz", global_topo_tiff_gz, mode = "wb")
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================

if(!file.exists(global_topo_tiff)){
    R.utils::gunzip(global_topo_tiff_gz, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
}


In [ ]:
# MOVE TO DATA DIRECTORY 
data.directory <- "data"
ifelse(!dir.exists(file.path(data.directory)),
        dir.create(file.path(data.directory)),
        "Directory Exists")

file.rename(from=global_topo_tiff_gz,
            to=paste(data.directory, global_topo_tiff_gz, sep = "/"))
file.rename(from=global_topo_tiff,
            to=paste(data.directory, global_topo_tiff, sep = "/"))

## Depth (Bathymery)

In [ ]:
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster")


In [ ]:
library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics
library(dplyr)


In [ ]:
# 2.2 DOWNLOAD AND LOAD BATHYMETRY ----
options(timeout = 600)  # 10 minutes
bathy_file <- "global_topo_1min_topo_19_1.nc"
if(file.exists(bathy_file)){
    cat(bathy_file, "is (are) already in your repertory.")
} else {
    download.file("https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc", bathy_file, mode = "wb")
    print('File Downloaded')
}


In [ ]:
# MOVE TO DATA DIRECTORY 
data.directory <- "data"
ifelse(!dir.exists(file.path(data.directory)),
        dir.create(file.path(data.directory)),
        "Directory Exists")

file.rename(from=bathy_file,
            to=paste(data.directory, bathy_file, sep = "/"))
bathy_file <- paste(data.directory, bathy_file, sep = "/")

In [ ]:
# Open the NetCDF file
nc_file <- nc_open(bathy_file)


In [ ]:
# Get coordinates variables
longitude <- ncvar_get(nc_file,"lon")
latitude <- ncvar_get(nc_file,"lat")
z <- ncvar_get(nc_file,"z")
fillvalue <- ncatt_get(nc_file, "z", "_FillValue") # The fill value (aka, the no data value) is -9999.


In [ ]:
nc_close(nc_file)

In [ ]:
z[z == fillvalue$value] <- NA
nc.slice.min80 <- z[,1]

### Plots

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
Bathy <- rast(bathy_file)
print("Converted into Raster File")

png <- 1
if (png) {
  png(filename = "outputs/collection/Fig1-Bathy.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/collection/collection/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()


### Convert NetCDF to CSV
Based on [Copernicus Marine Services](https://help.marine.copernicus.eu/en/articles/6328012-how-to-convert-netcdf-to-csv-using-r)

In [ ]:
#  Extract the coordinates
nc_file <- nc_open(bathy_file)

dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)

In [ ]:
coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))
names(depth_matrix) <- c("lon", "lat", "depth")

nc_close(nc_file)

In [ ]:
output.directory <- "outputs/collection"
ifelse(!dir.exists(file.path(output.directory)),
        dir.create(file.path(output.directory)),
        "Directory Exists")

table_filename <- paste(output.directory,"netcdf_depth.csv", sep="/")
write.table(depth_matrix, table_filename, row.names=FALSE, sep="\t")

print("The matrix was exported in CSV format.")

# NASA Data - SST

In [ ]:
# 2.2 DOWNLOAD NASA NetCDF ----
system("python NASA_API.py")

In [ ]:
# 2.2 ANALYSIS ----
SST_file <- "data/2008/AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.nc"
file.exists(SST_file)

In [ ]:
# Open the NetCDF file

nc_sst_file <- nc_open(SST_file)

#  Extract the coordinates
dim_sst_lon <- ncvar_get(nc_sst_file, "lon")
dim_sst_lat <- ncvar_get(nc_sst_file, "lat")
dim_sst <- ncvar_get(nc_sst_file, "sst")

sst_raw <- ncvar_get(nc_sst_file, "sst", raw_datavals = TRUE)


In [ ]:
# Get attributes
fill_value <- ncatt_get(nc_sst_file, "sst", "_FillValue")$value
scale      <- ncatt_get(nc_sst_file, "sst", "scale_factor")$value
sst_raw[sst_raw == fill_value] <- NA
sst <- sst_raw * scale


In [ ]:
# Group SST and coordinates in a table/ matrix
sst_coords <- expand.grid(dim_sst_lon, dim_sst_lat)
sst_matrix <- cbind(sst_coords, as.vector(sst))
names(sst_matrix) <- c("lon", "lat", "sst")
sst_matrix <- na.omit(sst_matrix)


In [ ]:
# Close file
nc_close(nc_sst_file)

In [ ]:
table_filename <- paste(output.directory,"AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.No_NA.csv", sep="/")
write.table(sst_matrix, table_filename, row.names=FALSE, sep="\t")


# Ice concentration

In [ ]:
library(ncdf4)
library(lubridate)
library(RColorBrewer)
library(lattice)


In [ ]:
# 2.2 DOWNLOAD COPERNICUS NetCDF ----

# Sea ice area & Sea ice concentration estimates as retrieved by the algorithm,
# and that were edited away by the various filters
# system("python API_Copernicus.py")

# OR

# Use 'Copernicus marine data' output as an input
ice_netcdf <- "./data/osisaff_obs-si_glo_phy_sic-south_my_amsr_cdr_P1D-m_1778859355280.nc"
# ice_netcdf <- 'galaxy_inputs/Datasets/Copernicus marine data.netcdf'
nc_iceC_file <- nc_open(ice_netcdf)


In [ ]:
#  Extract the coordinates
dim_ice_lon <- ncvar_get(nc_iceC_file, "longitude") # nc_iceC_file$dim[[3]]$vals . longitude
dim_ice_lat <- ncvar_get(nc_iceC_file, "latitude") # nc_iceC_file$dim[[2]]$vals . latitude
dim_ice_time <- ncvar_get(nc_iceC_file, "time") # nc_iceC_file$dim[[1]]$vals . time
dim_ice <- ncvar_get(nc_iceC_file, "ice_conc")
dim_raw_ice <- ncvar_get(nc_iceC_file, "raw_ice_conc_values")



In [ ]:
t_units <- ncatt_get(nc_iceC_file, "time", "units")

#sea_water_potential_concentration
T <- c("ice_conc", "raw_ice_conc_values")

# convert time -- split the time units string into fields
t_ustr <- strsplit(t_units$value, " ")
t_dstr <- strsplit(unlist(t_ustr)[3], "-")
date <- ymd(t_dstr) + dseconds(dim_ice_time)

T_array_ice <- dim_ice
# Quick map plot

# set the time step
t <- 1 #temperature on 2003-01-01
T_slice <- T_array_ice[,,t]


In [ ]:
image(dim_ice_lon, dim_ice_lat, T_slice, col = rev(brewer.pal(10,"RdBu")))

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/collection/IceConcentration.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/collection/IceConcentration.pdf")
}

# Plot Ice Concentration
image(x = dim_ice_lon, y = dim_ice_lat, z = T_slice, col = rev(brewer.pal(10,"RdBu")))

# Save the plot ---
dev.off()

In [ ]:
# Create a set of lonxlat pairs of values, one for each element in the Temp_array
grid <- expand.grid(lon=dim_ice_lon, lat=dim_ice_lat)  

cutpts <- c(12,13,14,15,16,17,18,19,20)   # set colorbar
levelplot(T_slice ~ lon * lat,
          data=grid, region=TRUE,
          pretty=T, at=cutpts, cuts=9,
          col.regions=(rev(brewer.pal(9,"RdBu"))), contour=0,
          xlab = "Longitude", ylab = "Latitude",
          main = "Sea Water Potential Temperature (°C)"
          )

In [ ]:
ice_raw <- ncvar_get(nc_iceC_file, "ice_conc", raw_datavals = TRUE)

# Get attributes
fill_value <- ncatt_get(nc_iceC_file, "ice_conc", "_FillValue")$value
scale      <- ncatt_get(nc_iceC_file, "ice_conc", "scale_factor")$value

ice_raw[ice_raw == fill_value] <- NA
ice <- ice_raw * scale


In [ ]:
ice_coords <- expand.grid(dim_ice_lon, dim_ice_lat,date)
ice_matrix <- cbind(ice_coords, as.vector(dim_ice))
names(ice_matrix) <- c("lon", "lat", "time", "iceC")


In [ ]:
# Function to replace NaN with column median
df_clean <- lapply(ice_matrix, function(iceC) {
    if (is.numeric(iceC)) {
      # Compute median excluding NA and NaN
      med <- median(iceC, na.rm = TRUE)
      
      # Replace NaN values with median
      iceC[is.nan(iceC)] <- med
    }
    return(iceC)}
)

In [ ]:
nc_close(nc_iceC_file)

In [ ]:
table_filename <- paste(output.directory,"Ice_CopernicusMarineData.csv", sep="/")
write.table(ice_matrix, table_filename, row.names=FALSE, sep="\t")

# Concatenate the Dataframes

In [ ]:
# ----------------
# Filter Out the outerbound values

## 1. Depth
Antarctic_depth_coords <- depth_matrix %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60
  )

## 2. SST
Antarctic_sst_coords <- sst_matrix %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60
  )

## 3. IceConc (Ice Concentration)
Antarctic_iceC_coords <- as.data.frame(df_clean) %>%
  dplyr::mutate(
    month = format(time, "%m") # Apply month format
  ) %>%
  dplyr::filter(
    lon >= 30, lon <= 150, lat >= -70, lat <= -60,
    (month %in% c("01", "02", "03", "12")) | (month == "04" & format(time, "%d") <= "01")
  ) %>%
    dplyr::group_by(lon, lat) %>% # Groups ONLY BY COORDINATES
    dplyr::summarise(
    ice_mean = mean(iceC, na.rm = TRUE), # Compute the average Ice concentration in summer
    .groups = "drop"
  )


In [ ]:
# ----------------
# Interpolation
## 1. Create a new grid
new.grid.lon <- seq(from=30.0, to=150.0, by=0.05)
new.grid.lat <- seq(from=-70.0, to=-60.0, by=0.05)
new.grid <- expand.grid(lon=new.grid.lon, lat=new.grid.lat)

## 2. Add Bathymetry (depth) + Sea Surface Temperature (sst) + IceConc (iceC)
### +  Fill the added columns  
new.grid$sst <- NA
new.grid$depth <- NA
new.grid$iceC <- NA


In [ ]:
system("conda install conda-forge::r-fnn")


In [ ]:
library(data.table)
library(FNN)

In [ ]:
grid   <- as.data.table(new.grid)
sst_dt <- as.data.table(Antarctic_sst_coords)
depth_dt <- as.data.table(Antarctic_depth_coords)
ice_dt <- as.data.table(Antarctic_iceC_coords)

In [ ]:
## SST
# Find nearest neighbor for each grid point
nn <- get.knnx(
  data  = as.matrix(sst_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)

# Assign interpolated values
grid[, sst := sst_dt$sst[nn$nn.index]]

In [ ]:
## DEPTH
# Find nearest neighbor for each grid point
nn <- get.knnx(
  data  = as.matrix(depth_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)

# Assign interpolated values
grid[, depth := depth_dt$depth[nn$nn.index]]

In [ ]:
### ICE CONCENTRATION
# Find nearest neighbor for each grid point
nn <- get.knnx(  data  = as.matrix(ice_dt[, .(lon, lat)]),  query = as.matrix(grid[, .(lon, lat)]),  k = 1)

# Assign interpolated values
grid[, iceC := ice_dt$ice_mean[nn$nn.index]]
grid %>% rename("ice_mean" = "iceC")

In [ ]:
table_filename <- paste(output.directory,"PelagicData_noNA_sst.csv", sep="/")
write.table(grid, table_filename, row.names=FALSE, sep="\t")

# Statistical Analyses

In [ ]:
system("conda install conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools")


In [ ]:
install.packages("remotes")
remotes::install_github("bioRgeo/bioregion")

# install.packages("devtools")
# devtools::install_github("bioRgeo/bioregion")

In [ ]:
library(cluster)
library(data.table)
library(bioregion)

In [ ]:
# Default value of dissimilarity matrix

# -----------------------
# clean SST flag
# grid[sst == -32767, sst := NA]
# -----------------------

rdm.rows <- sample(nrow(grid), 900, replace = FALSE)
X <- grid[rdm.rows, .(sst, depth, iceC)]

# compute Gower distances (small subset!)
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)

In [ ]:
clara.object <- bioregion::nhclu_clara(
    dissimilarity = distance.matrix, n_clust = c(50, 100, 200))
                                       


In [ ]:
clusters900 <- grid[rdm.rows, ]
clusters900$K_50 <- clara.object$clusters$K_50
clusters900$K_100 <- clara.object$clusters$K_100
clusters900$K_200 <- clara.object$clusters$K_200


In [ ]:
df_K50 <- clusters900 %>%
  mutate(across(starts_with("K_"), as.integer)) %>%
  group_by(K_50) %>%
  summarise(
    mean_sst   = mean(sst, na.rm = TRUE),
    mean_depth = mean(depth, na.rm = TRUE),
    mean_iceC  = mean(iceC, na.rm = TRUE)
  )

df_K100 <- clusters900 %>%
    mutate(across(starts_with("K_"), as.integer)) %>%
    group_by(K_100) %>%
  summarise(
    mean_sst   = mean(sst, na.rm = TRUE),
    mean_depth = mean(depth, na.rm = TRUE),
    mean_iceC  = mean(iceC, na.rm = TRUE)
  )

df_K200 <- clusters900 %>%
  mutate(across(starts_with("K_"), as.integer)) %>%
  group_by(K_200) %>%
  summarise(
    mean_sst   = mean(sst, na.rm = TRUE),
    mean_depth = mean(depth, na.rm = TRUE),
    mean_iceC  = mean(iceC, na.rm = TRUE)
  )

In [ ]:
system("conda install conda-forge::r-phangorn")

library(phangorn)

In [ ]:
X <- as.data.table(df_K50)[, .(mean_sst, mean_depth, mean_iceC)]

# compute Gower distances
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)
rownames(gower_matrix) <- colnames(gower_matrix)
colnames(gower_matrix)[1] <- paste0("\t", colnames(gower_matrix)[1])


In [ ]:
hclust.pelagic.K50 <- phangorn::upgma(D = gower_matrix, method = "ward.D2")


In [ ]:
table_filename <- paste(output.directory,"PelagicData_noNA_sst_df_K50.txt", sep="/")
write.table(gower_matrix, table_filename, row.names=TRUE, col.names = TRUE, sep="\t") # Only format accepted by Build a UPGMA tree Galaxy tool

table_filename <- paste(output.directory,"PelagicData_noNA_sst_df_K50clust.txt", sep="/")


In [ ]:
X <- as.data.table(df_K100)[, .(mean_sst, mean_depth, mean_iceC)]

# compute Gower distances
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)
rownames(gower_matrix) <- colnames(gower_matrix)
colnames(gower_matrix)[1] <- paste0("\t", colnames(gower_matrix)[1])


In [ ]:
hclust.pelagic.K100 <- phangorn::upgma(D = gower_matrix, method = "ward.D2")

plot(as.phylo(hclust.pelagic.K100), cex = 0.6, label.offset = 0.5, main = "UPGMA Tree")
print(hclust.pelagic.K100)


In [ ]:
table_filename <- paste(output.directory,"PelagicData_noNA_sst_df_K100.txt", sep="/")
write.table(gower_matrix, table_filename, row.names=TRUE, col.names = TRUE, sep="\t")

table_filename <- paste(output.directory,"PelagicData_noNA_sst_df_K100clust.txt", sep="/")


In [ ]:
X <- as.data.table(df_K200)[, .(mean_sst, mean_depth, mean_iceC)]

# compute Gower distances
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)
rownames(gower_matrix) <- colnames(gower_matrix)
colnames(gower_matrix)[1] <- paste("\t", colnames(gower_matrix)[1])


In [ ]:
hclust.pelagic.K200 <- phangorn::upgma(D = gower_matrix, method = "ward.D2")

plot(as.phylo(hclust.pelagic.K200), cex = 0.6, label.offset = 0.5, main = "UPGMA Tree")
print(hclust.pelagic.K200)


In [ ]:
table_filename <- paste(output.directory,"PelagicData_noNA_sst_df_K200.txt", sep="/")
write.table(gower_matrix, table_filename, row.names=TRUE, col.names = TRUE, sep="\t")

table_filename <- paste(output.directory,"PelagicData_noNA_sst_df_K200clust.txt", sep="/")
